# Reproducing Logic-LM (Pan et al., Findings of EMNLP 2023): clean run

Runs the authors' symbolic-solver stage on their saved GPT-4 logic programs (dev split), 5 datasets × 3 seeds.
CPU runtime, no API key, about 10 minutes in total. This version already includes every fix described in `NOTES.md`.
My original session, with all the failed attempts, is in `logs/colab_session_with_attempts.ipynb`.

In [ ]:
# 1. Clone
!git clone https://github.com/suriyakala-gif/Logic-LLM.git
%cd Logic-LLM
!mkdir -p logs results
!( date -u; python --version; head -2 /etc/os-release; nproc; free -h | head -2 ) 2>&1 | tee logs/00_environment_fresh.log

In [ ]:
# 2. Install the 8 solver packages (pinned) and make Pyke work on Python 3.12+
!pip install -r requirements.txt 2>&1 | tail -1
import site, os
open(os.path.join(site.getsitepackages()[0], "zombie_imp_autoload.pth"), "w").write("import zombie_imp\n")
!chmod +x models/symbolic_solvers/Prover9/bin/*
!python -c "import z3, constraint, nltk, ply, func_timeout; from pyke import knowledge_engine; print('all solvers OK')"

In [ ]:
# 3. Run 5 datasets x 3 seeds with a ~4 GB memory cap (about 6 minutes)
!ulimit -v 4000000; python run_reproduction.py --seeds 0 1 2 --split dev --model gpt-4 2>&1 | tee logs/fresh_clone_run.log

In [ ]:
# 4. Comparison table and Figure 1 (paper values already in results/paper_values.csv)
!python make_table.py
!python make_figure.py --gray
from IPython.display import Image, display
display(Image("fig1_gray.png"))